# SPEC figure code — every panel

One cell per figure, each running that figure's scripts in order and showing the
panels it writes. Run the whole notebook, or just the cell for the figure you
care about — the cells are independent, apart from figure 5's internal ordering.

Run `00_getting_started.ipynb` first if you have not: it checks the install and
the data in a few seconds, and every failure mode shows up there more clearly
than here.

Roughly 15 minutes end to end. Nothing is written into the input tree; every
panel and source-data file lands under `SPEC_OUTPUT_ROOT`, printed below.


## Paths

Set these two if the data or the figures live outside the repository. Both are
read once, when `spec_config` is imported in the next cell, so **run this cell
first**; if you change it later, restart the kernel.


In [ ]:
import os

# Where the deposited search outputs are, and where figures are written.
#
#   DATA_ROOT   holds figure1/input, figure2/input, ... exactly as deposited
#   OUTPUT_ROOT receives every PDF, PNG and _sourcedata.csv, plus the caches
#
# Leave both as None to use the repository itself: unpack the deposited archive
# so that figure2/input/ sits beside figure2/scripts/, and figures land in
# <repository>/output/, which is git-ignored. Nothing is ever written into the
# input tree either way.
#
# To keep them elsewhere, give absolute paths, for example
#   DATA_ROOT   = r'D:\SPEC_data'
#   OUTPUT_ROOT = r'D:\SPEC_figures'

DATA_ROOT = None      # None -> leave as the environment has it
OUTPUT_ROOT = None    # None -> leave as the environment has it

for _name, _value in (('SPEC_DATA_ROOT', DATA_ROOT),
                      ('SPEC_OUTPUT_ROOT', OUTPUT_ROOT)):
    if _value is not None:
        os.environ[_name] = str(_value)
    print(f'{_name:18s} {os.environ.get(_name, "(unset: repository default)")}')


In [ ]:
import os
import subprocess
import sys
import time
from pathlib import Path

import matplotlib.image as mpimg
import matplotlib.pyplot as plt

# The repository root is the nearest parent holding spec_config.py, so the
# notebook works whether it is opened from notebooks/ or from the root.
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'spec_config.py').exists())
sys.path.insert(0, str(REPO))
import spec_config as cfg


def show(path, width=8.0):
    """Draw a saved panel PNG inline at its own aspect ratio."""
    img = mpimg.imread(path)
    h, w = img.shape[:2]
    fig, ax = plt.subplots(figsize=(width, width * h / w))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(Path(path).name, fontsize=9)
    plt.show()
    # Without this the panel appears twice: plt.show() renders it, and the
    # inline backend renders every still-open figure again at the end of the
    # cell.
    plt.close(fig)


def run_panel(figure, script, display=True, width=8.0):
    """Run one panel script in a fresh interpreter and show what it wrote.

    A separate process, not %run: each script is written to be run standalone,
    and this keeps one script's globals out of the next one's.
    """
    path = REPO / figure / 'scripts' / script
    if not path.exists():
        raise FileNotFoundError(path)
    out_dir = Path(cfg.OUTPUT_ROOT) / figure
    started = time.time()
    # UTF-8 on both sides, explicitly. On Windows a piped child defaults to the
    # ANSI code page, so a panel script printing 'log10' as a subscript would
    # otherwise die on the write, or arrive undecodable here.
    env = dict(os.environ, PYTHONIOENCODING='utf-8')
    proc = subprocess.run([sys.executable, path.name], cwd=path.parent, env=env,
                          capture_output=True, encoding='utf-8',
                          errors='replace')
    print(proc.stdout, end='')
    if proc.returncode != 0:
        print(proc.stderr[-3000:], file=sys.stderr)
        raise RuntimeError(f'{figure}/{script} exited {proc.returncode}')
    fresh = sorted(p for p in out_dir.glob('*.png')
                   if p.stat().st_mtime >= started - 1)
    print(f'\n[{figure}/{script}] {time.time() - started:.0f}s, '
          f'{len(fresh)} panel(s) written to {out_dir}')
    if display:
        for p in fresh:
            show(p, width=width)
    return fresh


def run_figure(figure, scripts, width=8.0):
    """Run every script of one figure, in the given order."""
    written = []
    for script in scripts:
        written += run_panel(figure, script, width=width)
    return written


In [ ]:
print('repository :', REPO)
print('data root  :', cfg.DATA_ROOT)
print('output root:', cfg.OUTPUT_ROOT)


## Figure 1

Digestion phases (c, d) and the two LC front ends (e, f).

In [ ]:
_ = run_figure('figure1', [
    'panel_cd_phase_comparison.py',
    'panel_e_lc_comparison.py',
    'panel_f_lc_correlation.py',
])

## Figure 2

Digestion kinetics, sample volume, input titration and detergent tolerance.

In [ ]:
_ = run_figure('figure2', [
    'panel_a_digestion_kinetics.py',
    'panel_b_sample_volume.py',
    'panel_c_pg_vs_input.py',
    'panel_d_detergent_tolerance.py',
])

## Figure 3

mTRAQ labelling (b, c) and the SPEC fractionation (e-h).

In [ ]:
_ = run_figure('figure3', [
    'panel_b_labeling_efficiency.py',
    'panel_c_channel_overlap.py',
    'panel_ef_species_counts.py',
    'panel_gh_coverage_peptides.py',
])

## Figure 4

FFPE tissue (b-e) and mouse organs (f, g).

In [ ]:
_ = run_figure('figure4', [
    'panel_bcde_ffpe_spec.py',
    'panel_fg_organs.py',
])

## Figure 5

Single muscle fibers. **This one is a pipeline and must run in order**: `01` and `02` write the caches every later step reads, and `01` runs directLFQ over all 164 fibers, so it is the slowest cell in the notebook.

In [ ]:
_ = run_figure('figure5', [
    '01_load_and_filter.py',
    '02_fiber_types.py',
    '05_pca.py',
    '07_marker_ranks.py',
    '08_id_counts.py',
    '11_composition_strip.py',
    '12_fiber_type_composition.py',
    '13_regulated_abundance.py',
    '14_volcano_I_vs_IIb.py',
])

## Figure 6

Plasma glycoproteomics (b-d) and the ubiquitinome (f-h).

In [ ]:
_ = run_figure('figure6', [
    'panel_bd_glyco.py',
    'panel_c_glyco_depth.py',
    'panel_fg_kgg.py',
    'panel_h_pathway_coverage.py',
])

## Supplementary figure 1

Peptide overlap, digestion completeness and physicochemical bias.

In [ ]:
_ = run_figure('supplementary_figure1', [
    'supplement_peptide_overlap.py',
    'supplement_mc0_by_phase.py',
    'supplement_phase_properties.py',
])

## Supplementary figure 2

Front-end hydrophobicity bias, and two preparations two months apart. Reads `supplementary_figure1`'s input tree.

In [ ]:
_ = run_figure('supplementary_figure2', [
    'supplement_lc_hydrophobicity.py',
    'supplement_reproducibility.py',
])

## Supplementary figure 3

Peptide-level counts behind figure 2, plus the protease titration. Reads `figure2`'s input tree.

In [ ]:
_ = run_figure('supplementary_figure3', [
    'panel_d_peptides_kinetics.py',
    'panel_e_peptides_volume.py',
    'panel_f_peptides_input.py',
    'panel_g_peptides_detergent.py',
    'supplement_protease_load.py',
])

## Supplementary figure 4

Signal recovery and organ separation for the FFPE preparations.

In [ ]:
_ = run_figure('supplementary_figure4', [
    'panel_ab_total_intensity.py',
    'panel_c_pca_organs.py',
    'panel_d_digestion_completeness.py',
])

## Supplementary figure 5

Digestion efficiency and membrane coverage across the single fibers. The two `prep_*` scripts must run first; they are provenance for inputs that are also deposited.

In [ ]:
_ = run_figure('supplementary_figure5', [
    'prep_digestion_efficiency.py',
    'prep_membrane_input.py',
    'supplement_digestion_and_membrane.py',
])

## Supplementary figure 6

The unenriched plasma proteome, before glycopeptide enrichment.

In [ ]:
_ = run_figure('supplementary_figure6', [
    'panel_ab_plasma_proteome.py',
])

## What this notebook does not run

Three of the repository's 46 scripts are left out, so the cells above cover 43:

- `figure2/scripts/panel_a_workflow_comparison.py` — a qualitative comparison,
  superseded by the schematic in the paper.
- `figure2/scripts/workflow_timeline.py` — a schematic, not a data panel.
- `figure5/scripts/15_outlier_filter_sensitivity.py` — the reproducibility check.
  It re-runs figure 5's whole pipeline with the technical-outlier filter disabled
  and reports how far the regulated set, the fold changes and the marker calls
  move, which would roughly double this notebook's runtime.

All three are runnable; use `run_panel` on them directly, e.g.

```python
_ = run_panel('figure5', '15_outlier_filter_sensitivity.py')
```
